# Lecture 10: Missingness Mechanisms
**DATA 2201 — Foundations of Data Science**  
**Fall 2026**

## Lecture Outline
- Missing data and their implications for analysis
- Missing by design (MD), MCAR, MAR, and MNAR
- MCAR demo: Permutation test with a height dataset
- MAR demo: Permutation test with the same dataset (if time permits)
- Homework: Fast permutation tests

## 1. Introduction to Missing Data

Datasets often contain missing values because information was not collected, was lost, or was not provided. In pandas, these values commonly appear as `NaN`.


### Why Does Missing Data Matter?

Suppose five students have exam scores:

**40, 50, 70, 80, 90**

The actual class average is **66**.

Now suppose students with low scores choose not to report them. If **40 and 50** are missing, the available scores are:

**70, 80, 90**

Their average is **80**, making class performance appear better than it actually is.

## 2. Missingness Mechanisms

A **missingness mechanism** describes why values are missing. Understanding the reason helps us assess potential bias.

### 2.1 Missing by Design (MD)

**Definition:** A value is intentionally not collected under the survey or study design.

**Examples:**
- People without a car skip the question about car color.
- Unemployed respondents skip the question about their current employer.

**Key point:** These questions are **not applicable**.

### 2.2 Missing Completely at Random (MCAR)

**Definition:** Missingness is unrelated to observed information or the missing value itself.

**Examples:**
- A few randomly selected survey forms are accidentally lost.
- A scale randomly fails to save weights, regardless of the weight measured.

**Key point:** Missingness does not systematically favor particular people or values.

### 2.3 Missing at Random (MAR)

**Definition:** After accounting for observed information, missingness does not additionally depend on the missing value.

**Examples:**
- One clinic loses more weight records because of a software issue; within each clinic, loss is unrelated to weight.
- One class section loses more exam scores because of an upload issue; within each section, loss is unrelated to score.

**Key point:** The recorded **clinic or class section** explains the different chances of missingness.

### 2.4 Missing Not at Random (MNAR; also NMAR)

**Definition:** Missingness still depends on the missing value after accounting for observed information.

**Examples:**
- People with higher incomes are more likely to leave income blank, even among those with similar recorded jobs.
- Students with lower scores are more likely to withhold their scores, even within the same class section.

**Key point:** The missing value itself influences whether it is reported.

# Formal definitions

We won't spend much time on these in lecture, but you may find them helpful.

Suppose we have:
- A dataset $Y$ contains **observed values** ($Y_{\text{obs}}$) and **missing values** ($Y_{\text{mis}}$).

For example, consider the recorded heights are **170, 165, missing, and 180 cm**:
- $Y_{\text{obs}}$ contains the known heights: **170, 165, and 180 cm**.
- $Y_{\text{mis}}$ represents the **unknown height** of the person whose measurement is missing.

1. Data is **missing completely at random** (MCAR) if 

$$\text{P}(\text{data is present} \: | \: Y_{obs}, Y_{mis}) = \text{P}(\text{data is present})$$

In other words, the chance that a missing value appears doesn't depend on our data (missing or otherwise).

2. Data is **missing at random** (MAR) if 

$$\text{P}(\text{data is present} \: | \: Y_{obs}, Y_{mis}) = \text{P}(\text{data is present} \: | \: Y_{obs})$$

That is, MAR data is **actually MCAR**, **conditional** on $Y_{obs}$.

3. Data is **not missing at random** (NMAR) if  

$$\text{P}(\text{data is present} \: | \: Y_{obs}, Y_{mis})$$

cannot be simplified. That is, in NMAR data, **missingness is dependent on the missing value** itself.

## Required Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_inline.backend_inline import set_matplotlib_formats
from IPython.display import display, IFrame, HTML
from pathlib import Path

import plotly
import plotly.figure_factory as ff
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.options.plotting.backend = "plotly"

## 3. Exploring Missingness in Height Data

We use adult children’s and parents’ heights (in inches) to simulate and compare **MCAR** and **MAR** missingness.

**Data file:** `data/midparent.csv`. Run the cells in order.

Here, we know the mechanisms because we create the missingness. In real data, assessing them requires knowledge of the collection process and assumptions.

In [ ]:
heights_path = Path('data') / 'midparent.csv'
heights = (pd.read_csv(heights_path)
           .rename(columns={'childHeight': 'child'})
           [['father', 'mother', 'gender', 'child']])
heights.head()

### 3.1 Initial Data Inspection
Check whether the dataset contains missing values before introducing artificial missingness.

In [ ]:
heights.isna().sum()

## 4. Simulating MCAR Missingness
A random sample of 30% of the rows is selected, and the corresponding child heights are removed. Selection does not depend on child height, parent heights, or gender.

In [ ]:
np.random.seed(42) # So that we get the same results each time (for lecture).

heights_mcar = heights.copy()
idx = heights_mcar.sample(frac=0.3).index
heights_mcar.loc[idx, 'child'] = np.nan

In [ ]:
heights_mcar.isna().mean()

### 4.1 Defining the Missingness Groups
- Each row of `heights_mcar` belongs to one of two **groups**:
    - Group 1: `'child'` is missing.
    - Group 2: `'child'` is not missing.

In [ ]:
heights_mcar['child_missing'] = heights_mcar['child'].isna()
heights_mcar.head()

### 4.2 Comparing null and non-null `'child'` distributions for `'gender'`
The table gives the gender distribution within each missingness group. Each column sums to 1. Differences between these distributions indicate an association between gender and missingness.

In [ ]:
gender_dist = (
    heights_mcar
    .assign(child_missing=heights_mcar['child'].isna())
    .pivot_table(index='gender', columns='child_missing', aggfunc='size')
)

# Added just to make the resulting pivot table easier to read.
gender_dist.columns = ['child_missing = False', 'child_missing = True']

gender_dist = gender_dist / gender_dist.sum()
gender_dist

In [ ]:
gender_dist.plot(kind='barh', 
                 title='Gender by Missingness of Child Height', 
                 barmode='group')

### 4.3 Permutation Test of Association
We test whether gender is associated with missingness of child height.

- **Null hypothesis ($H_0$):** Gender and child-height missingness are independent.
- **Alternative hypothesis ($H_1$):** Gender and child-height missingness are associated.

The test statistic is the **total variation distance** between the two gender distributions:

$$\operatorname{TVD}(p,q)=\frac{1}{2}\sum_i |p_i-q_i|.$$

Larger values indicate greater differences. With two categories, TVD equals the absolute difference in the proportion belonging to either category.

Permuting gender labels preserves the group sizes while removing the observed association with missingness. Repeated permutations produce a reference distribution under the null hypothesis.

#### Simulation

The code to run our simulation largely looks the same as in previous permutation tests. Remember, we are comparing two distributions:
- The distribution of 'gender' when 'child' is missing.
- The distribution of 'gender' when 'child' is not missing.

In [ ]:
n_repetitions = 500
shuffled = heights_mcar.copy()

tvds = []
for _ in range(n_repetitions):
    
    # Shuffling genders. 
    # Note that we are assigning back to the same DataFrame for performance reasons; 
    # see lec10-fast-permutation-tests.html.
    shuffled['gender'] = np.random.permutation(shuffled['gender'])
    
    # Computing and storing the TVD.
    pivoted = (
        shuffled
        .pivot_table(index='gender', columns='child_missing', aggfunc='size')
    )
    
    pivoted = pivoted / pivoted.sum()
    
    tvd = pivoted.diff(axis=1).iloc[:, -1].abs().sum() / 2
    tvds.append(tvd)

### Computer observed TVD

In [ ]:
observed_tvd = gender_dist.diff(axis=1).iloc[:, -1].abs().sum() / 2
observed_tvd

### Plot the simulated TVD distribution and mark the observed TVD with a red vertical line.

In [ ]:
fig = px.histogram(pd.DataFrame(tvds), x=0, nbins=50, histnorm='probability', 
                   title='Empirical Distribution of the TVD')
fig.add_vline(x=observed_tvd, line_color='red')
fig.add_annotation(text=f'<span style="color:red">Observed TVD = {round(observed_tvd, 2)}</span>',
                   x=2.3 * observed_tvd, showarrow=False, y=0.16)
fig.update_layout(yaxis_range=[0, 0.2])

### Compare Simulated and Observed TVD

In [ ]:
(np.array(tvds) >= observed_tvd).mean()

### 4.4 Interpreting the Test

The estimated **p-value is 0.838**: 83.8% of the permuted TVDs were at least as large as the observed TVD.

Since $0.838 > 0.05$, we **fail to reject the null hypothesis**. There is insufficient evidence that child-height missingness is associated with gender.

This result is consistent with MCAR, but does not prove it. Here, we know the mechanism is MCAR because we removed child heights randomly.

### 4.5 Comparing Father-Height Distributions
Compare father heights between the missing and observed groups using a **kernel density estimate (KDE)**, a smoothed representation of a numerical distribution. We use the same plot in the MAR example to make the comparison consistent.

In [ ]:
# Used for plotting examples.
def create_kde_plotly(df, group_col, group1, group2, vals_col, title=''):
    fig = ff.create_distplot(
        hist_data=[df.loc[df[group_col] == group1, vals_col], df.loc[df[group_col] == group2, vals_col]],
        group_labels=[group1, group2],
        show_rug=False, show_hist=False,
        colors=['#ef553b', '#636efb'],
    )
    return fig.update_layout(title=title)

In [ ]:
plt.figure(figsize=(8, 4))
sns.kdeplot(
    data=heights_mcar,
    x='father',
    hue='child_missing',
    common_norm=False
)
plt.xlabel("Father height (inches)")
plt.title("Father's Height by Missingness of Child Height")
plt.show()

## 5. Simulating MAR Missingness
Now, we will make `'child'` heights MAR by deleting `'child'` heights according to a random procedure that **depends on other columns**.

In [ ]:
np.random.seed(42) # So that we get the same results each time (for lecture).

def make_missing(r):
    rand = np.random.uniform() # Random real number between 0 and 1.
    if r['father'] > 72 and rand < 0.5:
        return np.nan
    elif r['gender'] == 'female' and rand < 0.3:
        return np.nan
    else:
        return r['child']
    
heights_mar = heights.copy()
heights_mar['child'] = heights_mar.apply(make_missing, axis=1)
heights_mar['child_missing'] = heights_mar['child'].isna()

### 5.1 Comparing null and non-null `'child'` distributions for `'gender'`, again
Compare gender composition and father-height distributions after introducing MAR missingness. Associations with these observed variables are expected because they enter the deletion rule.

In [ ]:
gender_dist = (
    heights_mar
    .assign(child_missing=heights_mar['child'].isna())
    .pivot_table(index='gender', columns='child_missing', aggfunc='size')
)

# Added just to make the resulting pivot table easier to read.
gender_dist.columns = ['child_missing = False', 'child_missing = True']

gender_dist = gender_dist / gender_dist.sum()
gender_dist

In [ ]:
gender_dist.plot(kind='barh', 
                 title='Gender by Missingness of Child Height (MAR Example)', 
                 barmode='group')

### 5.2 Comparing null and non-null `'child'` distributions for `'father'`, again

In [ ]:
plt.figure(figsize=(8, 4))
sns.kdeplot(
    data=heights_mar,
    x='father',
    hue='child_missing',
    common_norm=False
)
plt.xlabel("Father height (inches)")
plt.title("Father's Height by Missingness of Child Height (MAR Example)")
plt.show()

- The above two distributions look quite different.
    - This is because we artificially created missingness in the dataset in a way that depended on `'father'` and `'gender'`.

# If Time Permits: MAR Demo (Otherwise, Complete as Homework Practice1)

### Permutation Test: Father Height and Child-Height Missingness (MAR Demonstration)

We compare father heights between children whose heights are **missing** and **recorded**.

- **$H_0$:** Child-height missingness is independent of father height.
- **$H_1$:** Child-height missingness is associated with father height.

**Our test statistic is:**

$$
\text{Mean father height when child height is missing}
-
\text{Mean father height when child height is recorded}
$$

In [ ]:
def mean_diff(df):
    means = df.groupby('child_missing')['father'].mean()
    return means.loc[True] - means.loc[False]

observed_diff = mean_diff(heights_mar)
print(f"Observed difference: {observed_diff:.2f} inches")

#### Shuffle missingness labels 1,000 times to simulate mean differences under $H_0$.

In [ ]:
rng = np.random.default_rng(42)
simulated_diffs = []

for _ in range(1000):
    shuffled = heights_mar.assign(
        child_missing=rng.permutation(heights_mar['child_missing'])
    )
    simulated_diffs.append(mean_diff(shuffled))

simulated_diffs = np.array(simulated_diffs)

### Plot the null distribution, mark the observed difference in red.

In [ ]:
plt.hist(simulated_diffs, bins=30, edgecolor='white')
plt.axvline(observed_diff, color='red', label='Observed difference')
plt.xlabel('Mean father-height difference (inches)')
plt.ylabel('Number of simulations')
plt.legend()
plt.show()

### Calculate the two-sided p-value.

In [ ]:
p_value = (
    np.sum(np.abs(simulated_diffs) >= abs(observed_diff)) + 1
) / (len(simulated_diffs) + 1)
print(f"P-value: {p_value:.4f}")

Since $p = 0.001 < 0.05$, we reject $H_0$: missingness is associated with father height. This example is **MAR** because the deletion rule uses observed father height and gender, not the child’s actual height.

## Homework Practice2

Please complete the [**lec10-fast-permutation-tests.ipynb** notebook](https://mybinder.org/v2/gh/sujanmtu/data2201-fa26/main?urlpath=%2Fdoc%2Ftree%2Flec%2Flec10%2Flec10-fast-permutation-tests.ipynb) with a peer.

# See you next lecture!